# LangChain 메모리


**학습 목표**

> 1. LLM은 기본적으로 이전 대화를 자동으로 기억하지 않으며, 이전 메시지를 다시 입력해야 멀티턴 대화가 된다는 점을 이해한다.
> 2. `SystemMessage`, `HumanMessage`, `AIMessage`의 역할을 구분하고 메시지 리스트를 직접 관리한다.
> 3. `MessagesPlaceholder`를 사용해 프롬프트 안에 이전 대화 기록을 삽입한다.
> 4. `trim_messages`와 최근 N턴 보존 방식으로 토큰 증가를 제어한다.
> 5. `RunnableWithMessageHistory`로 세션별 대화 기록을 자동 관리하는 방법을 익힌다.
> 6. **대화 기록을 요약하고 사용자 프로필로 추출하는 메모리 워크플로우**를 구성한다.





> 참고: 예전 LangChain에서는 `ConversationBufferMemory`, `ConversationSummaryMemory`, `ConversationBufferWindowMemory`를 많이 사용했습니다. 현재 LangChain 1.x에서는 메시지 리스트, `MessagesPlaceholder`, `RunnableWithMessageHistory`, LangGraph 기반 메모리 관리 방식이 더 권장됩니다.


# 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [1]:
# 필요한 라이브러리 설치
%pip install -U langchain langchain-core langchain-openai python-dotenv pydantic


Note: you may need to restart the kernel to use updated packages.


## (2) 라이브러리 Import


In [2]:
import os
from pathlib import Path
from getpass import getpass
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_core.runnables.history import RunnableWithMessageHistory


## (3) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [3]:
load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 있음


## (4) 모델 준비

- 메모리 실습에서는 모델 호출보다 **이전 대화를 어떤 형태로 저장하고 다시 전달하는지**가 핵심


In [4]:
MODEL = 'gpt-4.1-mini'

model = ChatOpenAI(
    model=MODEL,
    timeout=60,
    max_retries=3,
)

parser = StrOutputParser()

model


ChatOpenAI(metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.7'}}, output_version=None, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000025F26F9ACF0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000025F26F9B770>, root_client=<openai.OpenAI object at 0x0000025F26492270>, root_async_client=<openai.AsyncOpenA

# 2. 메모리란?
- LLM은 한 번의 호출만 보면 상태가 없음. 즉, 첫 번째 호출에서 사용자가 이름을 말해도 두 번째 호출에서 그 이름을 자동으로 기억하지 않음
- 대화를 이어가려면 이전 메시지를 다음 호출에 다시 포함해야 함

In [5]:
first = model.invoke('안녕? 나는 핑크퐁이야.')
print(first.content)

print('='*80)

second = model.invoke('내 이름이 뭐라고?')
print(second.content)

안녕, 핑크퐁! 만나서 반가워. 오늘 기분은 어때? 무엇을 도와줄까?
죄송하지만, 사용자의 이름을 알 수 있는 정보가 없습니다. 알려주시면 기억할 수 있어요!


- 위 예제에서는 두 번째 호출에 첫 번째 대화가 포함되지 않았음 그래서 모델은 사용자의 이름을 안정적으로 알 수 없음

- 메모리의 핵심 :  **현재 질문 + 이전 대화 기록 -> 모델 입력** 

# 3. 메시지 타입

- LangChain 채팅 모델은 단순 문자열뿐 아니라 메시지 객체 리스트를 입력으로 받을 수 있음

| 타입 | 의미 | 예시 |
|---|---|---|
| `SystemMessage` | 모델에게 주는 역할과 규칙 | `너는 친절한 튜터다` |
| `HumanMessage` | 사용자가 보낸 메시지 | `내 이름은 민수야` |
| `AIMessage` | 모델의 답변 | `반가워요, 민수님` |


In [8]:
messages = [
    SystemMessage('너는 인공지능 튜터이다. 한국말로 짧게 대답해.'),
    HumanMessage('메모리가 왜 필요한지 한 문장으로 설명해줘.')
]

response = model.invoke(messages)

print(response.content)
print('응답 타입:', type(response).__name__)

메모리는 데이터를 저장하고 빠르게 처리하기 위해 필요합니다.
응답 타입: AIMessage


In [9]:
messages.append(response)
messages.append(HumanMessage('방금 설명을 초등학생도 이해할 수 있게 바꿔줘.'))

follow_up = model.invoke(messages)
print(follow_up.content)

메모리는 컴퓨터가 기억을 하면서 일을 잘 하게 도와줘요.


# 4. 메시지 리스트로 수동 메모리 만들기

- 가장 단순한 메모리는 메시지 리스트

    1. 시스템 메시지를 먼저 넣는다.
    2. 사용자의 새 질문을 `HumanMessage`로 추가한다.
    3. 모델 응답을 `AIMessage`로 리스트에 다시 추가한다.
    4. 다음 호출 때 전체 리스트를 다시 전달한다.


In [11]:
conversation = [
    SystemMessage("너는 수강생의 학습 계획을 돕는 튜터이다. 짧고 구체적으로 답하라"),
]


def chat(user_text: str) -> str:
    conversation.append(HumanMessage(content=user_text))   
    ai_message=model.invoke(conversation)
    conversation.append(ai_message)
    return ai_message.content


print(chat("나는 이번 주에 LangChain 메모리를 공부할 거야."))
print("-" * 80)
print(chat("내가 이번 주에 공부하려는 주제가 뭐였지?"))


좋아요! 이번 주에 LangChain 메모리를 이해하기 위해 하루에 1~2시간씩 문서 읽기와 간단한 예제 코드를 작성해보세요. 중간중간 실습하며 기능을 익히는 것이 중요합니다.
--------------------------------------------------------------------------------
이번 주에는 LangChain 메모리를 공부할 예정입니다.


In [12]:
for i, message in enumerate(conversation):
    print(f"{i}. {type(message).__name__}: {message.content[:80]}")

0. SystemMessage: 너는 수강생의 학습 계획을 돕는 튜터이다. 짧고 구체적으로 답하라
1. HumanMessage: 나는 이번 주에 LangChain 메모리를 공부할 거야.
2. AIMessage: 좋아요! 이번 주에 LangChain 메모리를 이해하기 위해 하루에 1~2시간씩 문서 읽기와 간단한 예제 코드를 작성해보세요. 중간중간 실습하며
3. HumanMessage: 내가 이번 주에 공부하려는 주제가 뭐였지?
4. AIMessage: 이번 주에는 LangChain 메모리를 공부할 예정입니다.


## [실습] 나만의 3턴 대화 만들기

아래 코드에서 시스템 메시지를 바꾸고, 세 번 이상 대화를 이어가 보세요.

예시 주제:

- 여행 일정 도우미
- 주식 공부 튜터
- 게임 NPC 설정 도우미
- 고객 상담 챗봇


In [13]:
conversation = [
    SystemMessage("너는 트레킹 여행사의 여행 일정 도우미다. 최소 100자 이상으로 구체적으로 답하라"),
]


def chat(user_text: str) -> str:
    conversation.append(HumanMessage(content=user_text))   
    ai_message=model.invoke(conversation)
    conversation.append(ai_message)
    return ai_message.content


print(chat("나는 카자흐스탄 트레킹 여행을 가고 싶어."))
print("-" * 80)
print(chat("뭐 어떻게 하라고? 다시 말해봐. 더 구체적으로."))
print("-" * 80)
print(chat("어떤 여행사를 이용하면 좋을지, 그리고 가성비 금액 구성. 더나아가서 그 여행사의 연령대까지 알아봐"))


카자흐스탄은 광활한 자연과 독특한 풍경으로 트레킹을 즐기기에 매우 좋은 여행지입니다. 먼저, 여행 일정을 계획할 때 방문할 주요 지역으로는 알타이 산맥, 카테츠 산, 그리고 시베리아 접경지역의 튜글라 강 주변 등이 있습니다. 알타이 산맥은 청정 자연과 다양한 야생 동식물, 그리고 전통적인 유목민 문화를 체험할 수 있어 트레킹 코스로 인기가 높습니다. 일반적으로 5~7일 정도의 일정을 잡고, 중급 이상의 체력과 준비가 필요합니다.

여행 시기는 6월에서 9월 사이가 가장 적합한데, 이 시기에는 날씨가 온화하고 트레킹 경로가 안전합니다. 일정 중에는 현지 가이드와 함께 베이스 캠프에서 하룻밤 묵으며 자연을 즐기고, 전통 카자흐 음식도 체험할 수 있습니다. 또한, 고산 지대인 만큼 충분한 산소와 수분 섭취, 그리고 고산병 대비가 필요하니 건강 상태를 꼼꼼히 체크하시기 바랍니다.

만약 좀 더 가벼운 트레킹을 원하신다면, 알마티 근처의 메데우 계곡이나 이식쿨 호수 주변의 짧은 코스도 추천드립니다. 이 구간들은 경치가 아름다우면서도 난이도가 낮아 초보자도 부담 없이 즐길 수 있습니다. 상세한 일정과 맞춤형 코스 구성도 가능하니 원하시는 여행 스타일과 기간, 체력 수준을 알려주시면 더욱 구체적인 계획을 도와드리겠습니다.
--------------------------------------------------------------------------------
알겠습니다. 카자흐스탄 트레킹 여행을 위해 다음과 같은 구체적인 일정을 추천드립니다.

1. 출발 전 준비
- 카자흐스탄 비자 및 여권 확인, 예방접종(필요 시) 완료
- 트레킹 장비 준비: 등산화, 방한복, 방풍자켓, 등산스틱, 배낭, 충분한 물과 간식 등
- 체력 준비를 위해 출발 1~2개월 전부터 걷기 운동이나 근력 운동을 꾸준히 하세요.

2. 여행 일정 예시 (7일 코스)
- 1일차: 알마티 도착, 숙소 체크인 및 도시 탐방 (일정 적응과 장비 최종 점검)
- 2일차: 알마티에서 차로 약 3시간 이동해 

# 5. 메모리 제어: 최근 대화만 남기기

- 대화 전체를 계속 넣으면 토큰 수가 늘어 비용과 지연 시간이 증가
- 가장 단순한 방법은 최근 N턴만 남기는 것



In [14]:
# 아래 함수는 시스템 메시지는 유지하고, 최근 대화 메시지만 남김
def keep_recent_messages(messages: list[BaseMessage], max_messages: int = 6) -> list[BaseMessage]:
    system_messages = [m for m in messages if isinstance(m, SystemMessage)]
    non_system_messages = [m for m in messages if not isinstance(m, SystemMessage)]
    return system_messages + non_system_messages[-max_messages:]


sample_history = [
    SystemMessage(content="너는 학습 튜터다."),
    HumanMessage(content="내 이름은 민수야."),
    AIMessage(content="반가워요, 민수님."),
    HumanMessage(content="나는 LangChain을 공부해."),
    AIMessage(content="좋습니다. 체인과 메모리부터 시작해 봅시다."),
    HumanMessage(content="오늘은 메모리를 배울 거야."),
    AIMessage(content="메시지 리스트와 세션 히스토리를 보면 됩니다."),
    HumanMessage(content="내 이름이 뭐였지?"),
]

recent_history = keep_recent_messages(sample_history, max_messages=4)

print(f"원본 메시지 수: {len(sample_history)}")
print(f"최근 메시지 수: {len(recent_history)}")
for message in recent_history:
    print(type(message).__name__, "-", message.content)


원본 메시지 수: 8
최근 메시지 수: 5
SystemMessage - 너는 학습 튜터다.
AIMessage - 좋습니다. 체인과 메모리부터 시작해 봅시다.
HumanMessage - 오늘은 메모리를 배울 거야.
AIMessage - 메시지 리스트와 세션 히스토리를 보면 됩니다.
HumanMessage - 내 이름이 뭐였지?


# 6. trim_messages
- `trim_messages`는 메시지 개수가 아니라 토큰 수를 기준으로 대화를 줄임 
- 실제 서비스에서는 최근 대화 보존과 토큰 기준 트리밍을 함께 사용


In [31]:
from langchain_core.messages.utils import trim_messages

trimmed_history = trim_messages(
    sample_history, 
    max_tokens=50,          # 남길 전체 대화의 최대 토큰 수
    strategy="last",        # 최근 메시지 우선
    token_counter=model,
    include_system=True     # 시스템 메시지는 가능하면 유지
)

print(f"트리밍 전: {len(sample_history)}개")
print(f"트리밍 후: {len(trimmed_history)}개")
for message in trimmed_history:
    print(f"  [{type(message).__name__:14}] {message.content[:30]}")

트리밍 전: 8개
트리밍 후: 3개
  [SystemMessage ] 너는 학습 튜터다.
  [AIMessage     ] 메시지 리스트와 세션 히스토리를 보면 됩니다.
  [HumanMessage  ] 내 이름이 뭐였지?


# 7. MessagesPlaceholder
- `MessagesPlaceholder`는 프롬프트 템플릿 안에 이전 대화가 들어갈 자리를 만드는 기능

    - 시스템 역할과 현재 입력을 템플릿으로 관리할 수 있다.
    - 이전 대화는 `history` 변수로 분리할 수 있다.
    - 메모리 관리 코드와 프롬프트 설계를 분리할 수 있다.


In [26]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

memory_prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 {role}.  한국어로 짧게 대답해.'),
    MessagesPlaceholder(variable_name='history'),
    ('human','{input}')
])

memory_chain = memory_prompt | model | parser

history = [
    HumanMessage('초보자 보스 몬스터 한 마리 만들어주세요.'),
    AIMessage('이름: 졸린 곰탱이, hp 200, 공격은 느리지만 잠이 오는 스턴 광역기'),
]

result = memory_chain.invoke({
    'role': '게임 기획 멘토',
    'history': history,
    'input': '그 보스 약점도 하나 붙여줘. 초보가 잡기 좋을 만큼만'
})

print(result)


약점: 빛 속성 공격에 1.5배 데미지. 졸려서 빛에 약함!


# 8. RunnableWithMessageHistory
- 수동으로 메시지를 append하는 방식은 이해하기 쉽지만, 여러 사용자의 대화를 관리하기에는 번거로움
- `RunnableWithMessageHistory`는 체인을 감싸서 다음 작업을 자동화함

    1. `session_id`로 대화 기록을 찾는다.
    2. 이전 대화를 프롬프트의 `history`에 넣는다.
    3. 현재 사용자 입력과 모델 응답을 기록에 추가한다.


In [27]:
# 세션별로 대화 기록을 저장할 딕셔너리
session_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    # 처음 보는 session_id 라면 새 대화 기록 객체를 만듦
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()

    # 해당 session_id의 대화 기록을 반환
    return session_store[session_id]

session_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 수강생을 돕는 튜터다. 이전 대화를 기억해 답한다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

base_session_chain = session_prompt | model | parser

# 기본 체인에 메시지 히스토리 기능을 붙임
chat_with_history = RunnableWithMessageHistory(
    base_session_chain,             # 실제로 실행할 기본 체인
    get_session_history,            # session_id를 받아 해당 세션의 대화 기록을 반환하는 함수
    input_messages_key='input',     # 사용자의 현재 입력이 들어있는 key 이름
    history_messages_key='history'  # 프롬프트 안에서 이전 대화가 들어갈 key 이름
)

c:\Users\playdata2\work_space(playdata)\SKN30_playdata\LLM\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [28]:
# student-a 라는 세션 ID를 가진 설정값
config_a = {"configurable": {"session_id": "student-a"}}

# 첫 번째 대화
# session_id가 'student-a'인 대화 기록에 이 사용자 입력과 AI 응답이 저장
print(chat_with_history.invoke(
    {"input": "나는 오늘 랭체인 메모리를 배우고 있어"},
    config=config_a,
))

print("-" * 80)

print(chat_with_history.invoke(
    {"input": "내가 공부하는 주제는 뭐야?"},
    config=config_a,
))


랭체인 메모리를 배우고 있다니 좋네요! 랭체인 메모리는 대화의 문맥을 기억해서 더 자연스럽고 일관된 대화를 가능하게 하는 기술이에요. 혹시 랭체인 메모리 관련해서 궁금한 점이나 도움이 필요한 부분이 있을까요?
--------------------------------------------------------------------------------
당신이 오늘 공부하는 주제는 '랭체인 메모리'입니다. 추가로 궁금한 점 있으면 언제든 물어보세요!


In [29]:
config_b = {"configurable": {"session_id": "student-b"}}

print(chat_with_history.invoke(
    {"input": "나는 오늘 RAG를 공부하고 있어."},
    config=config_b,
))

print("-" * 80)

print(chat_with_history.invoke(
    {"input": "내가 공부하는 주제가 뭐라고?"},
    config=config_b,
))


RAG에 대해 공부하고 계시는군요! RAG는 Retrieval-Augmented Generation의 약자로, 외부 지식이나 문서에서 정보를 검색(retrieval)한 후 이를 바탕으로 답변을 생성(generation)하는 방법입니다. 주로 챗봇이나 질의응답 시스템에서 많이 활용되죠.

더 구체적으로 어떤 부분을 공부하고 계신가요? 예를 들어, RAG 아키텍처, 구현 방법, 또는 실습 코드 관련 내용 중 궁금한 점이 있으면 알려주세요!
--------------------------------------------------------------------------------
지금 공부하고 계신 주제는 RAG, 즉 Retrieval-Augmented Generation입니다.


In [42]:
for session_id, history_obj in session_store.items():
    print("=" * 80)
    print("session_id:", session_id)
    for message in history_obj.messages:
        print(type(message).__name__, "-", message.content[:100])


session_id: student-a
HumanMessage - 나는 오늘 랭체인 메모리를 배우고 있어
AIMessage - 랭체인 메모리를 배우고 있다니 좋네요! 랭체인 메모리는 대화의 문맥을 기억해서 더 자연스럽고 일관된 대화를 가능하게 하는 기술이에요. 혹시 랭체인 메모리 관련해서 궁금한 점이나 도
HumanMessage - 내가 공부하는 주제는 뭐야?
AIMessage - 당신이 오늘 공부하는 주제는 '랭체인 메모리'입니다. 추가로 궁금한 점 있으면 언제든 물어보세요!
session_id: student-b
HumanMessage - 나는 오늘 RAG를 공부하고 있어.
AIMessage - RAG에 대해 공부하고 계시는군요! RAG는 Retrieval-Augmented Generation의 약자로, 외부 지식이나 문서에서 정보를 검색(retrieval)한 후 이를 바
HumanMessage - 내가 공부하는 주제가 뭐라고?
AIMessage - 지금 공부하고 계신 주제는 RAG, 즉 Retrieval-Augmented Generation입니다.
session_id: customer-a
HumanMessage - 내 주문번호는 A-1004야. 배송이 늦어지고 있어.
AIMessage - 주문번호 A-1004의 배송 지연에 대해 불편을 드려 죄송합니다. 제가 배송 상황을 확인해보겠습니다. 잠시만 기다려 주세요.
HumanMessage - 내 주문번호가 뭐였지?
AIMessage - 고객님께서 알려주신 주문번호는 A-1004입니다. 배송 관련하여 더 도와드릴 사항이 있으면 말씀해 주세요.
HumanMessage - 내 주문번호는 A-1004야. 배송이 늦어지고 있어.
AIMessage - 주문번호 A-1004의 배송 지연 문제를 다시 확인하겠습니다. 현재 배송 상태를 점검한 후 가능한 빨리 안내해드리겠습니다. 조금만 기다려 주세요.
HumanMessage - 내 주문번호가 뭐였지?
AIMessage - 고객님께서 알려주신 주문번호는 A-1004입니다. 추가

In [43]:
for session_id, history_obj in session_store2.items():
    print("=" * 80)
    print("session_id:", session_id)
    for message in history_obj.messages:
        print(type(message).__name__, "-", message.content[:100])


session_id: customer-a
HumanMessage - 내 주문번호는 A-1004야. 배송이 늦어지고 있어.
AIMessage - 안녕하세요, 고객님. 주문번호 A-1004의 배송 지연으로 불편을 드려 죄송합니다. 현재 해당 주문의 배송 상태를 확인해보니, 택배 기사님의 일정이 지연되어 배송이 늦어지고 있는 
HumanMessage - 내 주문번호가 뭐였지?
AIMessage - 고객님께서 알려주신 주문번호는 A-1004입니다. 추가로 궁금한 점 있으시면 언제든 말씀해 주세요!
session_id: customer-b
HumanMessage - 내 주문번호는 B-1004야. 배송이 왜 이렇게 빨리 온 거야?
AIMessage - 안녕하세요, 고객님! 주문번호 B-1004의 배송이 예상보다 빨리 도착해 놀라셨죠? 고객님의 택배는 지역 물류센터의 원활한 처리 덕분에 평소보다 빠르게 출고되었습니다. 빠른 배송으
HumanMessage - 그건 정확한 답변이 아니라 CRM 센터한테 전달해서 나한테 연락하라고 해.
AIMessage - 알겠습니다, 고객님. 주문번호 B-1004 건에 대해 정확한 배송 관련 안내를 위해 CRM 센터로 전달해 드리겠습니다. CRM 센터에서 확인 후 빠른 시일 내에 고객님께 직접 연락


## [실습] 세션별 상담 챗봇 만들기

아래 코드를 수정해 `customer-a`, `customer-b`처럼 서로 다른 고객의 대화가 섞이지 않는지 확인해 보세요.


In [40]:
# 세션별로 대화 기록을 저장할 딕셔너리
session_store2: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    # 처음 보는 session_id 라면 새 대화 기록 객체를 만듦
    if session_id not in session_store2:
        session_store2[session_id] = InMemoryChatMessageHistory()

    # 해당 session_id의 대화 기록을 반환
    return session_store2[session_id]

session_prompt2 = ChatPromptTemplate.from_messages([
    ("system", "너는 택배 회사 상담원이다. 이전 대화를 기억해 답한다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

base_session_chain2 = session_prompt2 | model | parser

# 기본 체인에 메시지 히스토리 기능을 붙임
chat_with_history2 = RunnableWithMessageHistory(
    base_session_chain2,             # 실제로 실행할 기본 체인
    get_session_history,            # session_id를 받아 해당 세션의 대화 기록을 반환하는 함수
    input_messages_key='input',     # 사용자의 현재 입력이 들어있는 key 이름
    history_messages_key='history'  # 프롬프트 안에서 이전 대화가 들어갈 key 이름
)

c:\Users\playdata2\work_space(playdata)\SKN30_playdata\LLM\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [41]:
customer_a = {"configurable": {"session_id": "customer-a"}}
customer_b = {"configurable": {"session_id": "customer-b"}}

print(chat_with_history2.invoke(
    {"input": "내 주문번호는 A-1004야. 배송이 늦어지고 있어."},
    config=customer_a,
))

print(chat_with_history2.invoke(
    {"input": "내 주문번호는 B-1004야. 배송이 왜 이렇게 빨리 온 거야?"},
    config=customer_b,
))

print(chat_with_history2.invoke(
    {"input": "내 주문번호가 뭐였지?"},
    config=customer_a,
))

print(chat_with_history2.invoke(
    {"input": "그건 정확한 답변이 아니라 CRM 센터한테 전달해서 나한테 연락하라고 해."},
    config=customer_b,
))

안녕하세요, 고객님. 주문번호 A-1004의 배송 지연으로 불편을 드려 죄송합니다. 현재 해당 주문의 배송 상태를 확인해보니, 택배 기사님의 일정이 지연되어 배송이 늦어지고 있는 상황입니다. 정확한 배송 예정일은 오늘 오후까지 다시 안내드리겠습니다. 조금만 더 기다려 주시길 부탁드립니다. 추가 문의 사항 있으시면 언제든 말씀해 주세요.
안녕하세요, 고객님! 주문번호 B-1004의 배송이 예상보다 빨리 도착해 놀라셨죠? 고객님의 택배는 지역 물류센터의 원활한 처리 덕분에 평소보다 빠르게 출고되었습니다. 빠른 배송으로 불편함을 드리지 않도록 최선을 다하고 있습니다. 추가로 문의사항 있으시면 언제든 말씀해 주세요!
고객님께서 알려주신 주문번호는 A-1004입니다. 추가로 궁금한 점 있으시면 언제든 말씀해 주세요!
알겠습니다, 고객님. 주문번호 B-1004 건에 대해 정확한 배송 관련 안내를 위해 CRM 센터로 전달해 드리겠습니다. CRM 센터에서 확인 후 빠른 시일 내에 고객님께 직접 연락드리도록 하겠습니다. 불편을 드려 죄송합니다. 추가로 궁금한 사항 있으면 말씀해 주세요.


In [ ]:
trimmed_history2 = trim_messages(
    history_obj.messages, 
    max_tokens=100,          # 남길 전체 대화의 최대 토큰 수
    strategy="last",        # 최근 메시지 우선
    token_counter=model,
    include_system=True     # 시스템 메시지는 가능하면 유지
)

print(f"트리밍 전: {len(history_obj.messages)}개")
print(f"트리밍 후: {len(trimmed_history2)}개")
for message in trimmed_history2:
    print(f"  [{type(message).__name__:14}] {message.content[:30]}")

트리밍 전: 4개
트리밍 후: 2개
  [HumanMessage  ] 그건 정확한 답변이 아니라 CRM 센터한테 전달해서 나
  [AIMessage     ] 알겠습니다. 고객님의 주문번호 B-1004에 대한 빠른
